# IDC409: 01 Data Collection (Finance)


## Step 1: Install libraries

In [32]:
!pip install yfinance -q

## Step 2: Settings (change these if you like)

In [33]:
import pandas as pd, numpy as np, sqlite3

TICKERS = {"stock": "NVDA", "gold": "GC=F", "oil": "CL=F", "usdinr": "USDINR=X"}
START_DATE = "2021-01-01"

DB_PATH = "/content/market.db"
CSV_PATH = "/content/stocks_data_clean.csv"


## Step 3: Download prices (yfinance)
One column per asset, one row per day.

In [34]:
import yfinance as yf

raw = yf.download(list(TICKERS.values()), start=START_DATE, auto_adjust=True, progress=False)["Close"]
inv = {v: k for k, v in TICKERS.items()}
prices = raw.rename(columns=inv)[list(TICKERS.keys())]
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name = "date"

# Different markets have different holidays: carry the last known price forward
prices = prices.ffill().dropna()
print(prices.shape)
prices.tail()

(1516, 4)


Ticker,stock,gold,oil,usdinr
date,,,,
2026-10-26,216.000000,4038.399902,101.760002,94.938004
2026-10-27,217.940002,4074.600098,99.900002,95.186996
2026-10-28,215.380005,4051.600098,102.779999,95.177803
2026-10-29,213.660004,4031.399902,102.000000,95.209503
2026-10-30,213.660004,3993.699951,103.239998,95.135002


## Step 4: Save to SQLite
News is **appended** to whatever is already in the database, so re-running on another day grows your history.

In [35]:
conn = sqlite3.connect(DB_PATH)

# Prices: replace each time (always the full history)
prices.reset_index().to_sql("prices", conn, if_exists="replace", index=False)

print("Prices rows:", len(prices))
pd.read_sql("SELECT date, stock, gold FROM prices ORDER BY date DESC LIMIT 5", conn)

Prices rows: 1516


,date,stock,gold
0,2026-10-30 00:00:00,213.660004,3993.699951
1,2026-10-29 00:00:00,213.660004,4031.399902
2,2026-10-28 00:00:00,215.380005,4051.600098
3,2026-10-27 00:00:00,217.940002,4074.600098
4,2026-10-26 00:00:00,216.000000,4038.399902


## Step 5: Daily features, merge and export
- News on weekends is moved to the **next trading day**, because that is when the market can react.
- Days with no news get zeros, and `news_covered` marks the dates where news data actually exists, so you can model only that period if you want.

In [36]:
returns = prices.pct_change().add_suffix("_ret")        # stock_ret, gold_ret, oil_ret, usdinr_ret
df = prices.join(returns).reset_index()
df = df.dropna().reset_index(drop=True)                 # drops the first row (no previous day)

df.to_csv(CSV_PATH, index=False)
df.to_sql("daily_features", conn, if_exists="replace", index=False)

print("Final table:", df.shape, "|", df["date"].min().date(), "->", df["date"].max().date())
df.tail()

Final table: (1515, 9) | 2021-01-05 -> 2026-10-30


Ticker,date,stock,gold,oil,usdinr,stock_ret,gold_ret,oil_ret,usdinr_ret
1510,2026-10-26,216.000000,4038.399902,101.760002,94.938004,0.023891,0.007610,-0.014622,0.002394
1511,2026-10-27,217.940002,4074.600098,99.900002,95.186996,0.008981,0.008964,-0.018278,0.002623
1512,2026-10-28,215.380005,4051.600098,102.779999,95.177803,-0.011746,-0.005645,0.028829,-0.000097
1513,2026-10-29,213.660004,4031.399902,102.000000,95.209503,-0.007986,-0.004986,-0.007589,0.000333
1514,2026-10-30,213.660004,3993.699951,103.239998,95.135002,0.000000,-0.009352,0.012157,-0.000782


## Step 6: Download your files


In [37]:
from google.colab import files
files.download(DB_PATH)
files.download(CSV_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>